# 01 — Quality control

Decides, in code, which participants and trials enter the analysis set, so that
the Methods section can cite a rule instead of a judgement call.

Reads `data/derived/` (the MATLAB output tables) and `data/raw/vr/` (read-only).
Writes `data/derived/qc_trial_inventory.csv` and `data/derived/qc_exclusions.csv`.

*Определяет кодом, кто и какие трайлы попадают в анализ, чтобы в Methods можно
было сослаться на правило, а не на решение «на глаз». Читает `data/derived/` и
`data/raw/vr/` (только на чтение), пишет два QC-файла в `data/derived/`.*

### Setup

Paths, acquisition constants and the QC thresholds. The thresholds live here and
nowhere else, so the text below can refer to them by name and cannot drift from
what the code actually uses.

*Пути, константы записи и пороги QC. Пороги заданы здесь и больше нигде, поэтому
текст ниже ссылается на имена, а не на значения, и не может разойтись с кодом.*

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DERIVED = REPO / "data" / "derived"
RAW_VR = REPO / "data" / "raw" / "vr"

SR = 90            # Hz after resampling
T_TRIAL = 210      # s, full recording
T_QUIET = 50       # s of quiet stance before the perturbation starts
CYCLE = 20         # s, one stimulus cycle

MIN_DURATION = 209.0    # s — a shorter recording is dropped
MAX_ZERO_FRAC = 0.5     # marker channel flat at zero for more than half the samples
MAX_CALIB_DRIFT = 0.10  # m — difference in mean HMD height between conditions

CONDITIONS = [f"{c}_t{t}" for c in ("HB", "LB") for t in (1, 2, 3, 4)]

### Participant list

`TrialOrder_Part2.xlsx` is the roster: one row per participant with body
measures and the file index used for each condition. Taking the IDs from there
rather than hard-coding them keeps the notebook honest if the roster changes.

*`TrialOrder_Part2.xlsx` — список участников с антропометрией и индексами файлов
по условиям. Берём ID оттуда, а не хардкодим, чтобы блокнот не разошёлся с
реальностью при изменении списка.*

In [2]:
order = pd.read_excel(DERIVED / "TrialOrder_Part2.xlsx")
PARTICIPANTS = order["Subject ID"].dropna().tolist()

print(f"{len(PARTICIPANTS)} participants x {len(CONDITIONS)} conditions "
      f"= {len(PARTICIPANTS) * len(CONDITIONS)} cells")

14 participants x 8 conditions = 112 cells


### Coverage of the analysis table

Input is the wide table, one row per participant and six sway parameters per
condition. Counting how many participant × condition cells carry a value with
`.notna()` over the eight `PeriodicPower` columns, and listing the ones that do not.

*На входе широкая таблица: строка на участника, шесть параметров на условие.
Считаем через `.notna()` по восьми колонкам `PeriodicPower`, в скольких ячейках
есть значение, и перечисляем пустые.*

In [3]:
balance = pd.read_csv(DERIVED / "balance_data_2026.csv")

coverage = (
    balance.set_index("ID")[[f"PeriodicPower {c}" for c in CONDITIONS]]
    .notna()
    .rename(columns=lambda c: c.replace("PeriodicPower ", ""))
)
empty = [(pid, cond) for pid, row in coverage.iterrows() for cond, ok in row.items() if not ok]

print(f"cells: {coverage.size} | filled: {coverage.size - len(empty)} | empty: {len(empty)}")
for pid, cond in empty:
    print(f"  empty: {pid} {cond}")

cells: 112 | filled: 107 | empty: 5
  empty: DO11UB30 HB_t1
  empty: DO11UB30 HB_t2
  empty: DO11UB30 HB_t3
  empty: DO11UB30 HB_t4
  empty: DO11UB30 LB_t4


### Is `rms_total` an independent measure?

`run_VRApp_analysis_2026.m` line 75 builds the resultant sway as
`sqrt(cdiff(com_ap).^2 + cdiff(com_ap).^2)` — the AP component twice, where AP
and ML were intended. If that is what ran, every `rms_total` is exactly
`sqrt(2) * rms_ap`. Testing it by dividing the two columns and comparing with
`np.allclose` rather than assuming.

*В строке 75 MATLAB-скрипта результирующее колебание собрано из `com_ap` дважды
вместо AP и ML. Если так, то `rms_total` тождественно равен `sqrt(2)·rms_ap` —
проверяем делением колонок и `np.allclose`, а не на слово.*

In [4]:
ratio = pd.concat(
    [balance[f"rms_total {c}"] / balance[f"rms_ap {c}"] for c in CONDITIONS]
).dropna()

print(f"rms_total / rms_ap over {len(ratio)} cells: "
      f"min {ratio.min():.6f}, max {ratio.max():.6f}, sqrt(2) = {np.sqrt(2):.6f}")
print("identical to sqrt(2) everywhere:", bool(np.allclose(ratio, np.sqrt(2))))

rms_total / rms_ap over 107 cells: min 1.414214, max 1.414214, sqrt(2) = 1.414214
identical to sqrt(2) everywhere: True


### Do repeated pipeline runs agree?

`VR-App_output.csv` is appended to on every run, so a trial processed more than
once appears more than once. Grouping by `fname` and counting distinct values per
column with `.nunique()` separates the two directly measured quantities from the
parameters that come out of the model fit.

*`VR-App_output.csv` дописывается при каждом прогоне, поэтому трайл может
встречаться несколько раз. Группируем по `fname` и считаем `.nunique()` по
колонкам, чтобы отделить измеренные величины от параметров подгонки модели.*

In [5]:
vr_log = pd.read_csv(DERIVED / "VR-App_output.csv")
by_trial = vr_log.groupby("fname")

measured = ["response sway power", "random sway power"]
fitted = ["visual Weight - W", "time delay - dt", "Loop Gain - Kp", "Kd",
          "Torque FB gain - Glp", "b", "sim Err"]

repeatability = pd.DataFrame({
    "trials with >1 value": [int((by_trial[c].nunique() > 1).sum()) for c in measured + fitted],
    "max spread": [
        by_trial[c].agg(
            lambda x: (x.max() - x.min()) / abs(x.mean()) if x.nunique() > 1 and x.mean() else np.nan
        ).max()
        for c in measured + fitted
    ],
}, index=measured + fitted)

print(f"{len(vr_log)} rows, {vr_log.fname.nunique()} unique trials, "
      f"repeat counts {sorted(map(int, vr_log.fname.value_counts().unique()))}\n")
print(repeatability.to_string(float_format=lambda v: f"{v:.1%}" if v == v else "—"))

387 rows, 107 unique trials, repeat counts [2, 3, 4, 6]

                      trials with >1 value  max spread
response sway power                      0         NaN
random sway power                        0         NaN
visual Weight - W                       79       16.0%
time delay - dt                         79        3.0%
Loop Gain - Kp                          79        1.3%
Kd                                      79        3.8%
Torque FB gain - Glp                    79      300.0%
b                                       79       12.6%
sim Err                                 78        0.1%


### Collapsing the repeats

`pcl_ICfit_ml.m` optimises with `GlobalSearch`, which is stochastic, so repeated
rows cannot simply be assumed identical. Collapsing them with
`.groupby("fname").agg(median)`: a no-op for anything the pipeline measured, and a
stable choice for anything it fitted.

*`pcl_ICfit_ml.m` оптимизирует стохастическим `GlobalSearch`, поэтому считать
повторы одинаковыми нельзя. Схлопываем через `.groupby("fname").agg(median)` —
для измеренных величин это ничего не меняет, для подогнанных даёт устойчивый выбор.*

In [6]:
numeric = vr_log.select_dtypes("number").columns
trial_log = (
    vr_log.groupby("fname", as_index=False)
    .agg({"ID": "first", **{c: "median" for c in numeric}})
)
print(f"{len(vr_log)} rows -> {len(trial_log)} trials after collapsing repeats by median")

387 rows -> 107 trials after collapsing repeats by median


### Raw trial inventory

One row per raw recording of a main-sample participant. Duration comes from the
last line of the file via `seek()` to the tail, marker health from the first 30 s
via `read_csv(nrows=...)` — neither needs a full pass over 0.9 GB, and neither
writes to `data/raw/`.

*Строка на каждую сырую запись участников основной выборки. Длительность берём из
последней строки файла через `seek()` в хвост, состояние маркеров — из первых
30 с через `read_csv(nrows=...)`. Полный проход по 0,9 ГБ не нужен, в `data/raw/`
ничего не пишется.*

In [7]:
FOLDER = re.compile(r"^s(?P<pid>[A-Z0-9\u00c4\u00d6\u00dc]+)_(?P<session>[AB])_(?P<cond>H1?B|LB1?)_*$")


def final_timestamp(path: Path, nbytes: int = 4096) -> float:
    """Time column of the last row, without reading the whole file."""
    with path.open("rb") as fh:
        fh.seek(max(0, path.stat().st_size - nbytes))
        tail = fh.read()
    return float(tail.splitlines()[-1].decode("utf-8", "replace").split(",", 1)[0])


records = []
for folder in sorted(RAW_VR.iterdir()):
    match = FOLDER.match(folder.name) if folder.is_dir() else None
    if not match or match["pid"] not in PARTICIPANTS:
        continue
    for path in sorted(folder.glob("*.csv")):
        head = pd.read_csv(path, nrows=30 * SR,
                           usecols=["time", "ypos", "shld_ypos", "hip_ypos"])
        records.append({
            "ID": match["pid"],
            "session": match["session"],
            "condition": "HB" if match["cond"].startswith("H") else "LB",
            "file": path.name,
            "duration_s": final_timestamp(path),
            "hmd_height_m": head.ypos.mean(),
            "shoulder_height_m": head.shld_ypos.mean(),
            "hip_height_m": head.hip_ypos.mean(),
            "shoulder_zero_frac": (head.shld_ypos == 0).mean(),
            "hip_zero_frac": (head.hip_ypos == 0).mean(),
        })

inventory = pd.DataFrame(records).sort_values(["ID", "condition", "file"]).reset_index(drop=True)
inventory.to_csv(DERIVED / "qc_trial_inventory.csv", index=False)
print(f"{len(inventory)} recordings from {inventory.ID.nunique()} participants")

117 recordings from 14 participants


### Two failure modes in the inventory

Filtering the inventory with boolean masks on `duration_s` and on the two
zero-fraction columns: a recording that stops early, and a marker channel that
was never written. The second one matters because `getCOM` needs both the hip and
the shoulder marker to reconstruct the centre of mass.

*Фильтруем инвентарь булевыми масками по `duration_s` и по долям нулей: запись,
оборвавшаяся раньше срока, и канал маркера, который вообще не писался. Второе
важно, потому что `getCOM` восстанавливает центр масс по маркерам таза и плеч.*

In [8]:
print(f"shorter than {MIN_DURATION} s:")
print(inventory[inventory.duration_s < MIN_DURATION]
      [["ID", "condition", "file", "duration_s"]].to_string(index=False))

flat = inventory[(inventory.shoulder_zero_frac > MAX_ZERO_FRAC)
                 | (inventory.hip_zero_frac > MAX_ZERO_FRAC)]
print("\nbody markers flat at zero:")
print(flat.groupby(["ID", "condition"])[["shoulder_zero_frac", "hip_zero_frac"]]
      .agg(["size", "mean"]).to_string())

shorter than 209.0 s:
      ID condition                                                       file  duration_s
BI20OE24        HB Screen_Balance and VR_1_sBI20OE24_B_H1B__t1_INCOMPLETE.csv  181.424744
BI20OE24        LB   Screen_Balance and VR_1_sBI20OE24_A_LB_t1_INCOMPLETE.csv  161.123047
CA04TU11        HB   Screen_Balance and VR_1_sCA04TU11_B_HB_t1_INCOMPLETE.csv   75.347960
PE16IN18        LB   Screen_Balance and VR_1_sPE16IN18_A_LB_t1_INCOMPLETE.csv  203.635376

body markers flat at zero:
                   shoulder_zero_frac      hip_zero_frac     
                                 size mean          size mean
ID       condition                                           
AN06AN18 HB                         5  0.0             5  1.0
         LB                         4  0.0             4  1.0
DO11UB30 HB                         4  1.0             4  1.0


### Calibration drift between the two sessions

The centre of mass is scaled by the measured marker heights, so a participant
whose VR floor calibration differs between sessions carries an offset in exactly
the contrast of interest. Comparing mean HMD height per condition with
`.groupby().unstack()` and taking the absolute difference.

*Центр масс масштабируется измеренными высотами маркеров, поэтому расхождение
калибровки между сессиями даёт смещение ровно в том контрасте, который нас
интересует. Сравниваем среднюю высоту шлема по условиям через
`.groupby().unstack()` и берём модуль разности.*

In [9]:
calibration = (
    inventory.groupby(["ID", "condition"]).hmd_height_m.mean().unstack()
    .assign(drift_m=lambda d: (d.HB - d.LB).abs())
    .sort_values("drift_m", ascending=False)
)
print(calibration.round(3).to_string())

condition     HB     LB  drift_m
ID                              
EL30AD28   1.321  1.647    0.325
BI20OE24   1.484  1.493    0.009
CH10AL22   1.826  1.818    0.008
AN07IE11   1.667  1.661    0.006
AN23BE15   1.701  1.708    0.006
CH11RE22   1.567  1.561    0.006
CH08TU30   1.727  1.732    0.005
DO11UB30   1.749  1.753    0.004
RE13ON18   1.688  1.685    0.003
DI16UB31   1.517  1.521    0.003
AN06AN18   0.781  0.783    0.003
KA14RE15   1.608  1.607    0.001
PE16IN18   1.568  1.568    0.001
CA04TU11   1.681  1.682    0.000


### Assembling the exclusion table

Each rule is applied here and stored with the reason it fired, so the exclusion
table in the paper is generated rather than typed. A trial-level rule only fires
for files the pipeline actually used — a recording that `TrialOrder_Part2.xlsx`
already routes around is not an exclusion.

*Каждое правило применяется здесь и сохраняется вместе с причиной срабатывания,
чтобы таблица исключений в статье генерировалась, а не набиралась руками.
Правило уровня трайла срабатывает только на файлы, которые пайплайн реально брал.*

In [10]:
exclusions = []

for pid in inventory.loc[inventory.hip_zero_frac > MAX_ZERO_FRAC, "ID"].unique():
    if inventory[inventory.ID == pid].hip_zero_frac.gt(MAX_ZERO_FRAC).all():
        exclusions.append({"level": "participant", "ID": pid, "condition": "", "file": "",
                           "reason": "hip marker flat at zero in every recording; "
                                     "centre of mass not computable"})

for _, r in calibration[calibration.drift_m > MAX_CALIB_DRIFT].iterrows():
    exclusions.append({"level": "participant", "ID": r.name, "condition": "", "file": "",
                       "reason": f"VR calibration differs by {r.drift_m:.2f} m between "
                                 f"conditions; sway scaling not comparable"})

used = set(trial_log.fname)
for _, r in inventory[inventory.duration_s < MIN_DURATION].iterrows():
    if r.file in used:
        exclusions.append({"level": "trial", "ID": r.ID, "condition": r.condition,
                           "file": r.file,
                           "reason": f"recording ends at {r.duration_s:.1f} s of {T_TRIAL} s; "
                                     f"resampling would extrapolate"})

exclusions = pd.DataFrame(exclusions)
exclusions.to_csv(DERIVED / "qc_exclusions.csv", index=False)
print(exclusions.to_string(index=False))

      level       ID condition                                                       file                                                                           reason
participant AN06AN18                                                                             hip marker flat at zero in every recording; centre of mass not computable
participant EL30AD28                                                                      VR calibration differs by 0.33 m between conditions; sway scaling not comparable
      trial BI20OE24        HB Screen_Balance and VR_1_sBI20OE24_B_H1B__t1_INCOMPLETE.csv                 recording ends at 181.4 s of 210 s; resampling would extrapolate
      trial BI20OE24        LB   Screen_Balance and VR_1_sBI20OE24_A_LB_t1_INCOMPLETE.csv                 recording ends at 161.1 s of 210 s; resampling would extrapolate
      trial PE16IN18        LB   Screen_Balance and VR_1_sPE16IN18_A_LB_t1_INCOMPLETE.csv                 recording ends at 203.6 s of 210 s; res

### Resulting analysis set

Subtracting the participant-level exclusions from the roster, then counting how
many of the survivors have data in both conditions — that subset is what a
within-subject contrast can actually use.

*Вычитаем исключённых участников из списка и считаем, у скольких из оставшихся
есть данные в обоих условиях: именно этот набор доступен внутрисубъектному
сравнению.*

In [11]:
dropped = set(exclusions.loc[exclusions.level == "participant", "ID"])
analysis_set = [p for p in PARTICIPANTS if p not in dropped]
paired = [p for p in analysis_set
          if coverage.loc[p, [f"HB_t{t}" for t in (1, 2, 3, 4)]].any()
          and coverage.loc[p, [f"LB_t{t}" for t in (1, 2, 3, 4)]].any()]

print(f"recruited                     {len(PARTICIPANTS)}")
print(f"after participant exclusions  {len(analysis_set)}  ({', '.join(sorted(dropped))} removed)")
print(f"with data in both conditions  {len(paired)}")
print(f"trial-level exclusions        {int((exclusions.level == 'trial').sum())}")

recruited                     14
after participant exclusions  12  (AN06AN18, EL30AD28 removed)
with data in both conditions  11
trial-level exclusions        3
